In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import expit, logsumexp

**Funciones de la guía 2.2** (dadas): `descenso_gradiente` y `newton`. Se detienen cuando $\|\nabla J\|<$ `tol` o al llegar a `max_iter`, y devuelven el punto final, el historial del costo y la cantidad de iteraciones.

In [ ]:
def descenso_gradiente(costo, gradiente, theta0, eta=0.1, max_iter=10000, tol=1e-6):
    """
    Minimiza J(θ) por descenso por gradiente.

    Parámetros:
    - costo: función θ -> J(θ)
    - gradiente: función θ -> ∇J(θ), de la misma forma que θ
    - theta0: punto inicial
    - eta: tasa de aprendizaje
    - max_iter: número máximo de iteraciones
    - tol: tolerancia sobre ||∇J||

    Retorna:
    - theta: punto final
    - cost_hist: J(θ) en cada iteración, empezando por θ0
    - n_iter: iteraciones realizadas
    """
    theta = np.array(theta0, dtype=float)  # copia: no modificar theta0
    cost_hist = [costo(theta)]

    for t in range(max_iter):
        grad = gradiente(theta)
        if np.linalg.norm(grad) < tol:
            return theta, cost_hist, t
        theta = theta - eta * grad
        cost_hist.append(costo(theta))

    warnings.warn(f"descenso_gradiente no convergió en {max_iter} iteraciones")
    return theta, cost_hist, max_iter


def newton(costo, gradiente, hessiana, theta0, max_iter=100, tol=1e-6):
    """
    Minimiza J(θ) por el método de Newton.

    Parámetros:
    - costo: función θ -> J(θ)
    - gradiente: función θ -> ∇J(θ), (p,)
    - hessiana: función θ -> ∇²J(θ), (p, p)
    - theta0: punto inicial (p,)
    - max_iter: número máximo de iteraciones
    - tol: tolerancia sobre ||∇J||

    Retorna:
    - theta: punto final (p,)
    - cost_hist: J(θ) en cada iteración, empezando por θ0
    - n_iter: iteraciones realizadas
    """
    theta = np.array(theta0, dtype=float)
    cost_hist = [costo(theta)]

    for t in range(max_iter):
        grad = gradiente(theta)  # (p,)
        if np.linalg.norm(grad) < tol:
            return theta, cost_hist, t
        theta = theta - np.linalg.solve(hessiana(theta), grad)  # (p,)
        cost_hist.append(costo(theta))

    warnings.warn(f"newton no convergió en {max_iter} iteraciones")
    return theta, cost_hist, max_iter

## Ejercicio 1

**GLM con enlace canónico.** Con $\eta=X\theta$ (con la columna de bias), la familia exponencial da $\mu=A'(\eta)$ y $\operatorname{Var}(y)\propto A''(\eta)$. Minimizamos la log-verosimilitud negativa promediada (sin los términos que no dependen de $\theta$):

$$
J(\theta)=\frac1n\sum_{i=1}^n\big[A(\eta_i)-y_i\eta_i\big],\qquad
\nabla J(\theta)=\frac1nX^\top(\mu-y),\qquad
\nabla^2J(\theta)=\frac1nX^\top WX,\quad W=\operatorname{diag}\big(A''(\eta_i)\big).
$$

| Familia | $A(\eta)$ | $\mu=A'(\eta)$ | $w=A''(\eta)$ |
|---|---|---|---|
| Bernoulli | $\log(1+e^\eta)$ | $\sigma(\eta)$ | $\mu(1-\mu)$ |
| Poisson | $e^\eta$ | $e^\eta$ | $\mu$ |
| Exponencial *(adicional)* | $-\log(-\eta)$, con $\eta<0$ | $-1/\eta$ | $\mu^2$ |

**IRLS.** En cada iteración, con $\mu$ y $W$ evaluados en el $\theta$ actual, se arma la pseudo-respuesta $z=\eta+W^{-1}(y-\mu)$ y se resuelve un problema de mínimos cuadrados ponderados:

$$
\theta^{(t+1)}=(X^\top WX)^{-1}X^\top Wz .
$$

Es exactamente el paso de Newton, escrito de otra forma (lo verificamos en el ejercicio 2c).

**Criterio de parada**, en las dos clases: $\|\nabla J(\theta)\|<$ `tol`, o `max_iter` con aviso.

La clase base `myGLM` tiene lo común: las funciones de la familia, el costo, el gradiente y la predicción. `myIRLS` y `myGLMGradientDescent` heredan de ella y solo definen `fit`.

In [ ]:
class myGLM:
    """
    Base de los GLM con enlace canónico: funciones de la familia, costo, gradiente y predicción.
    """
    def __init__(self, familia="bernoulli", max_iter=100, tol=1e-6):
        """
        Inicializa el modelo.

        Parámetros:
        - familia: "bernoulli", "poisson" o "exponencial"
        - max_iter: número máximo de iteraciones
        - tol: tolerancia sobre ||∇J||
        """

        if familia not in ("bernoulli", "poisson", "exponencial"):
            raise ValueError(f"familia no válida: {familia!r}")
        self.familia = familia
        self.max_iter = max_iter
        self.tol = tol

        # Resultados del ajuste
        self.theta = None  # (p+1,), theta[0] es el bias
        self.cost_hist = None  # costo en cada iteración
        self.n_iter_ = None  # iteraciones realizadas

    def agregar_bias(self, X):
        """
        Agrega una columna de unos a la izquierda.

        Parámetros:
        - X: (n, p)

        Retorna:
        - X_b: (n, p+1)
        """
        return np.column_stack([np.ones(X.shape[0]), X])

    def theta_inicial(self, X_b, y):
        """
        Punto inicial: θ = 0, salvo en la familia exponencial, que necesita η < 0
        (se toma el bias = -1 / media(y), el MLE sin variables).

        Parámetros:
        - X_b: (n, p+1), y: (n,)

        Retorna:
        - theta0: (p+1,)
        """
        theta0 = np.zeros(X_b.shape[1])
        if self.familia == "exponencial":
            theta0[0] = -1 / np.mean(y)
        return theta0

    def parametros_familia(self, eta):
        """
        Media μ = A'(η) y pesos w = A''(η) de la familia.

        Parámetros:
        - eta: (n,)

        Retorna:
        - mu: (n,)
        - w: (n,)
        """

        # TODO: μ y w según la familia (ver la tabla)

        if self.familia == "bernoulli":
            mu = ...  # (n,)
            w = ...  # (n,)
        elif self.familia == "poisson":
            mu = ...
            w = ...
        else:  # exponencial (adicional), requiere η < 0
            mu = ...
            w = ...
        return mu, w

    def costo(self, X_b, y, theta):
        """
        J(θ) = (1/n) Σ [A(η_i) - y_i η_i], con η = X θ.

        Parámetros:
        - X_b: (n, p+1), y: (n,), theta: (p+1,)

        Retorna:
        - J: float
        """

        # TODO: A(η) según la familia (en Bernoulli, estable con np.logaddexp)

        eta = ...  # (n,)
        if self.familia == "bernoulli":
            A = ...  # (n,)
        elif self.familia == "poisson":
            A = ...
        else:
            A = ...
        return ...

    def gradiente(self, X_b, y, theta):
        """
        ∇J(θ) = X^T (μ - y) / n.

        Parámetros:
        - X_b: (n, p+1), y: (n,), theta: (p+1,)

        Retorna:
        - grad: (p+1,)
        """

        # TODO: (p+1, n) @ (n,) -> (p+1,)

        mu, _ = ...
        return ...

    def predict(self, X):
        """
        Media estimada μ = A'(X θ) (en Bernoulli, la probabilidad de la clase 1).

        Parámetros:
        - X: (n, p)

        Retorna:
        - mu: (n,)
        """

        # TODO: μ en η = X_b θ

        mu, _ = ...
        return mu


class myIRLS(myGLM):
    """
    GLM ajustado con IRLS.
    """
    def fit(self, X, y):
        """
        Ajusta el modelo.

        Parámetros:
        - X: (n, p)
        - y: (n,)

        Retorna:
        - self
        """

        # TODO:
        # - En cada iteración: η, μ, w; si ||∇J|| < tol, terminar
        # - Pseudo-respuesta z = η + (y - μ) / w
        # - Mínimos cuadrados ponderados: (X^T W X) θ = X^T W z
        # - Guardar el costo; si se llega a max_iter, avisar

        X_b = self.agregar_bias(X)  # (n, p+1)
        theta = self.theta_inicial(X_b, y)  # (p+1,)
        cost_hist = [self.costo(X_b, y, theta)]
        n_iter = self.max_iter

        for t in range(self.max_iter):
            eta = ...  # (n,)
            mu, w = ...  # (n,), (n,)
            if np.linalg.norm(...) < self.tol:
                n_iter = t
                break
            z = ...  # (n,)
            # (p+1, n) @ ((n, 1) * (n, p+1)) -> (p+1, p+1); (p+1, n) @ (n,) -> (p+1,)
            theta = ...  # (p+1,)
            cost_hist.append(...)
        else:
            warnings.warn(f"IRLS no convergió en {self.max_iter} iteraciones")

        self.theta = theta
        self.cost_hist = cost_hist
        self.n_iter_ = n_iter
        return self


class myGLMGradientDescent(myGLM):
    """
    GLM ajustado con descenso por gradiente.
    """
    def __init__(self, familia="bernoulli", eta=0.1, max_iter=10000, tol=1e-6):
        """
        Inicializa el modelo.

        Parámetros:
        - familia: "bernoulli", "poisson" o "exponencial"
        - eta: tasa de aprendizaje
        - max_iter: número máximo de iteraciones
        - tol: tolerancia sobre ||∇J||
        """
        super().__init__(familia, max_iter, tol)
        self.eta = eta

    def fit(self, X, y):
        """
        Ajusta el modelo.

        Parámetros:
        - X: (n, p)
        - y: (n,)

        Retorna:
        - self
        """

        # TODO: descenso_gradiente con el costo y el gradiente de la clase base

        X_b = self.agregar_bias(X)  # (n, p+1)
        theta, cost_hist, n_iter = descenso_gradiente(
            ...,
            ...,
            ...,
            eta=self.eta,
            max_iter=self.max_iter,
            tol=self.tol,
        )

        self.theta = theta
        self.cost_hist = cost_hist
        self.n_iter_ = n_iter
        return self

##### Solución

In [ ]:
class myGLM:
    """
    Base de los GLM con enlace canónico: funciones de la familia, costo, gradiente y predicción.
    """
    def __init__(self, familia="bernoulli", max_iter=100, tol=1e-6):
        """
        Inicializa el modelo.

        Parámetros:
        - familia: "bernoulli", "poisson" o "exponencial"
        - max_iter: número máximo de iteraciones
        - tol: tolerancia sobre ||∇J||
        """

        if familia not in ("bernoulli", "poisson", "exponencial"):
            raise ValueError(f"familia no válida: {familia!r}")
        self.familia = familia
        self.max_iter = max_iter
        self.tol = tol

        # Resultados del ajuste
        self.theta = None  # (p+1,), theta[0] es el bias
        self.cost_hist = None  # costo en cada iteración
        self.n_iter_ = None  # iteraciones realizadas

    def agregar_bias(self, X):
        """
        Agrega una columna de unos a la izquierda.

        Parámetros:
        - X: (n, p)

        Retorna:
        - X_b: (n, p+1)
        """
        return np.column_stack([np.ones(X.shape[0]), X])

    def theta_inicial(self, X_b, y):
        """
        Punto inicial: θ = 0, salvo en la familia exponencial, que necesita η < 0
        (se toma el bias = -1 / media(y), el MLE sin variables).

        Parámetros:
        - X_b: (n, p+1), y: (n,)

        Retorna:
        - theta0: (p+1,)
        """
        theta0 = np.zeros(X_b.shape[1])
        if self.familia == "exponencial":
            theta0[0] = -1 / np.mean(y)
        return theta0

    def parametros_familia(self, eta):
        """
        Media μ = A'(η) y pesos w = A''(η) de la familia.

        Parámetros:
        - eta: (n,)

        Retorna:
        - mu: (n,)
        - w: (n,)
        """

        if self.familia == "bernoulli":
            mu = expit(eta)  # (n,)
            w = mu * (1 - mu)  # (n,)
        elif self.familia == "poisson":
            mu = np.exp(eta)
            w = mu
        else:  # exponencial (adicional), requiere η < 0
            mu = -1 / eta
            w = mu**2
        return mu, w

    def costo(self, X_b, y, theta):
        """
        J(θ) = (1/n) Σ [A(η_i) - y_i η_i], con η = X θ.

        Parámetros:
        - X_b: (n, p+1), y: (n,), theta: (p+1,)

        Retorna:
        - J: float
        """

        eta = X_b @ theta  # (n,)
        if self.familia == "bernoulli":
            A = np.logaddexp(0, eta)  # (n,)
        elif self.familia == "poisson":
            A = np.exp(eta)
        else:
            A = -np.log(-eta)
        return np.mean(A - y * eta)

    def gradiente(self, X_b, y, theta):
        """
        ∇J(θ) = X^T (μ - y) / n.

        Parámetros:
        - X_b: (n, p+1), y: (n,), theta: (p+1,)

        Retorna:
        - grad: (p+1,)
        """

        mu, _ = self.parametros_familia(X_b @ theta)
        return X_b.T @ (mu - y) / len(y)

    def predict(self, X):
        """
        Media estimada μ = A'(X θ) (en Bernoulli, la probabilidad de la clase 1).

        Parámetros:
        - X: (n, p)

        Retorna:
        - mu: (n,)
        """

        mu, _ = self.parametros_familia(self.agregar_bias(X) @ self.theta)
        return mu


class myIRLS(myGLM):
    """
    GLM ajustado con IRLS.
    """
    def fit(self, X, y):
        """
        Ajusta el modelo.

        Parámetros:
        - X: (n, p)
        - y: (n,)

        Retorna:
        - self
        """

        X_b = self.agregar_bias(X)  # (n, p+1)
        theta = self.theta_inicial(X_b, y)  # (p+1,)
        cost_hist = [self.costo(X_b, y, theta)]
        n_iter = self.max_iter

        for t in range(self.max_iter):
            eta = X_b @ theta  # (n,)
            mu, w = self.parametros_familia(eta)  # (n,), (n,)
            if np.linalg.norm(X_b.T @ (mu - y) / len(y)) < self.tol:
                n_iter = t
                break
            z = eta + (y - mu) / w  # (n,)
            # (p+1, n) @ ((n, 1) * (n, p+1)) -> (p+1, p+1); (p+1, n) @ (n,) -> (p+1,)
            theta = np.linalg.solve(X_b.T @ (w[:, None] * X_b), X_b.T @ (w * z))  # (p+1,)
            cost_hist.append(self.costo(X_b, y, theta))
        else:
            warnings.warn(f"IRLS no convergió en {self.max_iter} iteraciones")

        self.theta = theta
        self.cost_hist = cost_hist
        self.n_iter_ = n_iter
        return self


class myGLMGradientDescent(myGLM):
    """
    GLM ajustado con descenso por gradiente.
    """
    def __init__(self, familia="bernoulli", eta=0.1, max_iter=10000, tol=1e-6):
        """
        Inicializa el modelo.

        Parámetros:
        - familia: "bernoulli", "poisson" o "exponencial"
        - eta: tasa de aprendizaje
        - max_iter: número máximo de iteraciones
        - tol: tolerancia sobre ||∇J||
        """
        super().__init__(familia, max_iter, tol)
        self.eta = eta

    def fit(self, X, y):
        """
        Ajusta el modelo.

        Parámetros:
        - X: (n, p)
        - y: (n,)

        Retorna:
        - self
        """

        X_b = self.agregar_bias(X)  # (n, p+1)
        theta, cost_hist, n_iter = descenso_gradiente(
            lambda th: self.costo(X_b, y, th),
            lambda th: self.gradiente(X_b, y, th),
            self.theta_inicial(X_b, y),
            eta=self.eta,
            max_iter=self.max_iter,
            tol=self.tol,
        )

        self.theta = theta
        self.cost_hist = cost_hist
        self.n_iter_ = n_iter
        return self

---

## Ejercicio 2

Datos sintéticos con $n=500$ y dos variables $X\sim\mathcal N(0,I)$:

- **logística**: $y_i\sim\text{Bernoulli}(\sigma(\eta_i))$, con $\theta=(-0.5,\ 2,\ -1)$;
- **Poisson**: $y_i\sim\text{Poisson}(e^{\eta_i})$, con $\theta=(0.5,\ 0.3,\ -0.2)$.

In [ ]:
import statsmodels.api as sm

rng = np.random.default_rng(0)
n = 500
X = rng.normal(size=(n, 2))  # (n, 2)
X_b = np.column_stack([np.ones(n), X])  # (n, 3)

theta_real = {
    "bernoulli": np.array([-0.5, 2.0, -1.0]),
    "poisson": np.array([0.5, 0.3, -0.2]),
}
y = {
    "bernoulli": rng.binomial(1, expit(X_b @ theta_real["bernoulli"])),
    "poisson": rng.poisson(np.exp(X_b @ theta_real["poisson"])),
}
print("Proporción de unos (logística):", y["bernoulli"].mean())
print("Media de los conteos (Poisson):", y["poisson"].mean(), "| máximo:", y["poisson"].max())

#### a) Coeficientes vs. `GLM` de `statsmodels`

In [ ]:
familias_sm = {"bernoulli": sm.families.Binomial(), "poisson": sm.families.Poisson()}
eta_gd = {"bernoulli": 1.0, "poisson": 0.2}

ajustes = {}
for familia in ("bernoulli", "poisson"):
    irls = myIRLS(familia).fit(X, y[familia])
    gd = myGLMGradientDescent(familia, eta=eta_gd[familia]).fit(X, y[familia])
    glm = sm.GLM(y[familia], X_b, family=familias_sm[familia]).fit()
    ajustes[familia] = {"irls": irls, "gd": gd, "statsmodels": glm}

    print(f"{familia}: IRLS {irls.n_iter_} iteraciones, gradiente {gd.n_iter_} iteraciones")
    display(
        pd.DataFrame(
            {
                "real": theta_real[familia],
                "myIRLS": irls.theta,
                "myGLMGradientDescent": gd.theta,
                "statsmodels": glm.params,
            },
            index=["bias", "x1", "x2"],
        ).round(5)
    )

#### b) Convergencia, tiempo y calidad

El tiempo es el promedio de varias corridas (un solo ajuste dura milisegundos y la medición es ruidosa); puede variar de una computadora a otra.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
filas = []
for ax, familia in zip(axes, ("bernoulli", "poisson")):
    J_opt = ajustes[familia]["irls"].cost_hist[-1]
    for nombre in ("irls", "gd"):
        modelo = ajustes[familia][nombre]
        ax.semilogy(np.array(modelo.cost_hist) - J_opt + 1e-16, "o-" if nombre == "irls" else "-", label=nombre)

        clase = myIRLS if nombre == "irls" else myGLMGradientDescent
        kwargs = {} if nombre == "irls" else {"eta": eta_gd[familia]}
        inicio = time.perf_counter()
        for _ in range(10):
            clase(familia, **kwargs).fit(X, y[familia])
        tiempo = (time.perf_counter() - inicio) / 10

        filas.append({
            "familia": familia,
            "método": nombre,
            "iteraciones": modelo.n_iter_,
            "tiempo (ms)": 1000 * tiempo,
            "máx. |θ - θ_statsmodels|": np.abs(modelo.theta - ajustes[familia]["statsmodels"].params).max(),
        })
    ax.set_xscale("log")
    ax.set_xlabel("iteración")
    ax.set_ylabel("J(θ) - J(θ*)")
    ax.set_title(familia)
    ax.legend()
plt.tight_layout()
plt.show()

pd.DataFrame(filas)

**Para discutir:** ¿Por qué IRLS converge en tan pocas iteraciones? ¿Compensa que cada iteración sea más cara?

#### c) IRLS es el método de Newton

In [ ]:
modelo = ajustes["bernoulli"]["irls"]
y_b = y["bernoulli"]

def hessiana_logistica(theta):
    _, w = modelo.parametros_familia(X_b @ theta)  # (n,)
    return X_b.T @ (w[:, None] * X_b) / len(y_b)  # (3, 3)

theta_nt, hist_nt, it_nt = newton(
    lambda th: modelo.costo(X_b, y_b, th),
    lambda th: modelo.gradiente(X_b, y_b, th),
    hessiana_logistica,
    np.zeros(3),
)
print("IRLS:", modelo.theta.round(8), f"({modelo.n_iter_} iteraciones)")
print("Newton:", theta_nt.round(8), f"({it_nt} iteraciones)")
print("¿Coinciden los historiales del costo?", np.allclose(modelo.cost_hist, hist_nt))

#### d) *(Adicional)* Familia exponencial

Las clases ya incluyen la familia exponencial. Con el enlace canónico, $\eta=X\theta$ tiene que ser negativo para todas las observaciones ($\mu=-1/\eta>0$): por eso el punto inicial no es $\theta=0$ sino el bias $-1/\bar y$. Simulamos $y_i\sim\text{Exponencial}$ con media $\mu_i=-1/\eta_i$, con variables en $(-1,1)$ para que $\eta$ quede lejos de 0.

In [ ]:
X_exp = rng.uniform(-1, 1, size=(n, 2))  # (n, 2)
X_exp_b = np.column_stack([np.ones(n), X_exp])
theta_real_exp = np.array([-1.0, 0.2, -0.1])
y_exp = rng.exponential(-1 / (X_exp_b @ theta_real_exp))  # media μ_i = -1/η_i

irls_exp = myIRLS("exponencial").fit(X_exp, y_exp)
gd_exp = myGLMGradientDescent("exponencial", eta=0.5).fit(X_exp, y_exp)

# statsmodels: familia Gamma con enlace inverso (el canónico); la exponencial es Gamma con forma 1
# (statsmodels usa η = 1/μ, el opuesto de nuestro η = -1/μ). Avisa que ese enlace no garantiza
# μ > 0: es la misma restricción η < 0 de nuestra parametrización, así que ocultamos el aviso.
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    glm_exp = sm.GLM(y_exp, X_exp_b, family=sm.families.Gamma(sm.families.links.InversePower())).fit()

pd.DataFrame(
    {
        "real": theta_real_exp,
        "myIRLS": irls_exp.theta,
        "myGLMGradientDescent": gd_exp.theta,
        "-statsmodels": -glm_exp.params,
    },
    index=["bias", "x1", "x2"],
).round(4)

---

## Ejercicio 3

**Regresión softmax** con $K$ clases. Los parámetros son una matriz $\Theta$ de forma `(p+1, K)`, una columna por clase. Con $S=X\Theta$ de forma `(n, K)` (los puntajes) y $P=\operatorname{softmax}(S)$ por filas,

$$
P_{ik}=\frac{e^{S_{ik}}}{\sum_le^{S_{il}}},\qquad
J(\Theta)=-\frac1n\sum_{i=1}^n\log P_{i,y_i}=\frac1n\sum_{i=1}^n\Big[\log\sum_le^{S_{il}}-S_{i,y_i}\Big],\qquad
\nabla J(\Theta)=\frac1nX^\top(P-Y),
$$

con $Y$ la matriz *one-hot* de las etiquetas, de forma `(n, K)`. Para que no desborde:

- el softmax se calcula restando a cada fila de $S$ su máximo (no cambia el resultado);
- el costo usa `logsumexp` de `scipy.special`.

`descenso_gradiente` funciona igual con una matriz $\Theta$ en lugar de un vector: `np.linalg.norm` de una matriz es la norma de Frobenius, la raíz de la suma de los cuadrados de todas las entradas.

In [ ]:
class mySoftmaxRegression:
    """
    Regresión softmax (logística multiclase) por descenso por gradiente.
    """
    def __init__(self, eta=0.1, max_iter=10000, tol=1e-6):
        """
        Inicializa el modelo.

        Parámetros:
        - eta: tasa de aprendizaje
        - max_iter: número máximo de iteraciones
        - tol: tolerancia sobre ||∇J|| (norma de Frobenius)
        """

        self.eta = eta
        self.max_iter = max_iter
        self.tol = tol

        # Resultados del ajuste
        self.clases = None  # etiquetas de las clases (K,)
        self.Theta = None  # (p+1, K), la fila 0 es el bias
        self.cost_hist = None  # costo en cada iteración
        self.n_iter_ = None  # iteraciones realizadas

    def agregar_bias(self, X):
        """
        Agrega una columna de unos a la izquierda.

        Parámetros:
        - X: (n, p)

        Retorna:
        - X_b: (n, p+1)
        """
        return np.column_stack([np.ones(X.shape[0]), X])

    def softmax(self, S):
        """
        Softmax por filas, estable.

        Parámetros:
        - S: puntajes (n, K)

        Retorna:
        - P: probabilidades (n, K), cada fila suma 1
        """

        # TODO: restar el máximo de cada fila antes de exponenciar; normalizar por filas

        S = ...  # (n, K)
        E = ...  # (n, K)
        return ...  # (n, K)

    def costo(self, X_b, indices, Theta):
        """
        J(Θ) = (1/n) Σ_i [logsumexp(S_i) - S_{i, y_i}], con S = X Θ.

        Parámetros:
        - X_b: (n, p+1)
        - indices: índice de la clase de cada observación (n,), valores en 0, ..., K-1
        - Theta: (p+1, K)

        Retorna:
        - J: float
        """

        # TODO: logsumexp por filas menos el puntaje de la clase correcta

        S = ...  # (n, K)
        puntaje_correcto = ...  # (n,)
        return ...

    def gradiente(self, X_b, Y, Theta):
        """
        ∇J(Θ) = X^T (P - Y) / n.

        Parámetros:
        - X_b: (n, p+1)
        - Y: one-hot (n, K)
        - Theta: (p+1, K)

        Retorna:
        - grad: (p+1, K)
        """

        # TODO: (p+1, n) @ (n, K) -> (p+1, K)

        P = ...  # (n, K)
        return ...

    def fit(self, X, y):
        """
        Ajusta el modelo partiendo de Θ = 0.

        Parámetros:
        - X: (n, p)
        - y: etiquetas (n,)

        Retorna:
        - self
        """

        # TODO:
        # - Clases e índice de la clase de cada observación (np.unique con return_inverse)
        # - Matriz one-hot Y (n, K)
        # - descenso_gradiente partiendo de Θ = 0 de forma (p+1, K)

        self.clases, indices = ...  # (K,), (n,)
        X_b = self.agregar_bias(X)  # (n, p+1)
        Y = ...  # (n, K)

        Theta, cost_hist, n_iter = descenso_gradiente(
            ...,
            ...,
            ...,
            eta=self.eta,
            max_iter=self.max_iter,
            tol=self.tol,
        )

        self.Theta = Theta
        self.cost_hist = cost_hist
        self.n_iter_ = n_iter
        return self

    def predict_proba(self, X):
        """
        Probabilidades de cada clase.

        Parámetros:
        - X: (n, p)

        Retorna:
        - P: (n, K)
        """

        # TODO

        return ...  # (n, K)

    def predict(self, X):
        """
        Clase de mayor probabilidad.

        Parámetros:
        - X: (n, p)

        Retorna:
        - y_pred: etiquetas (n,)
        """

        # TODO: índice de la clase de mayor probabilidad, traducido a la etiqueta

        return ...  # (n,)

##### Solución

In [ ]:
class mySoftmaxRegression:
    """
    Regresión softmax (logística multiclase) por descenso por gradiente.
    """
    def __init__(self, eta=0.1, max_iter=10000, tol=1e-6):
        """
        Inicializa el modelo.

        Parámetros:
        - eta: tasa de aprendizaje
        - max_iter: número máximo de iteraciones
        - tol: tolerancia sobre ||∇J|| (norma de Frobenius)
        """

        self.eta = eta
        self.max_iter = max_iter
        self.tol = tol

        # Resultados del ajuste
        self.clases = None  # etiquetas de las clases (K,)
        self.Theta = None  # (p+1, K), la fila 0 es el bias
        self.cost_hist = None  # costo en cada iteración
        self.n_iter_ = None  # iteraciones realizadas

    def agregar_bias(self, X):
        """
        Agrega una columna de unos a la izquierda.

        Parámetros:
        - X: (n, p)

        Retorna:
        - X_b: (n, p+1)
        """
        return np.column_stack([np.ones(X.shape[0]), X])

    def softmax(self, S):
        """
        Softmax por filas, estable.

        Parámetros:
        - S: puntajes (n, K)

        Retorna:
        - P: probabilidades (n, K), cada fila suma 1
        """

        S = S - S.max(axis=1, keepdims=True)  # (n, K)
        E = np.exp(S)  # (n, K)
        return E / E.sum(axis=1, keepdims=True)  # (n, K)

    def costo(self, X_b, indices, Theta):
        """
        J(Θ) = (1/n) Σ_i [logsumexp(S_i) - S_{i, y_i}], con S = X Θ.

        Parámetros:
        - X_b: (n, p+1)
        - indices: índice de la clase de cada observación (n,), valores en 0, ..., K-1
        - Theta: (p+1, K)

        Retorna:
        - J: float
        """

        S = X_b @ Theta  # (n, K)
        puntaje_correcto = S[np.arange(len(indices)), indices]  # (n,)
        return np.mean(logsumexp(S, axis=1) - puntaje_correcto)

    def gradiente(self, X_b, Y, Theta):
        """
        ∇J(Θ) = X^T (P - Y) / n.

        Parámetros:
        - X_b: (n, p+1)
        - Y: one-hot (n, K)
        - Theta: (p+1, K)

        Retorna:
        - grad: (p+1, K)
        """

        P = self.softmax(X_b @ Theta)  # (n, K)
        return X_b.T @ (P - Y) / len(Y)

    def fit(self, X, y):
        """
        Ajusta el modelo partiendo de Θ = 0.

        Parámetros:
        - X: (n, p)
        - y: etiquetas (n,)

        Retorna:
        - self
        """

        self.clases, indices = np.unique(y, return_inverse=True)  # (K,), (n,)
        X_b = self.agregar_bias(X)  # (n, p+1)
        Y = np.eye(len(self.clases))[indices]  # (n, K)

        Theta, cost_hist, n_iter = descenso_gradiente(
            lambda T: self.costo(X_b, indices, T),
            lambda T: self.gradiente(X_b, Y, T),
            np.zeros((X_b.shape[1], len(self.clases))),
            eta=self.eta,
            max_iter=self.max_iter,
            tol=self.tol,
        )

        self.Theta = Theta
        self.cost_hist = cost_hist
        self.n_iter_ = n_iter
        return self

    def predict_proba(self, X):
        """
        Probabilidades de cada clase.

        Parámetros:
        - X: (n, p)

        Retorna:
        - P: (n, K)
        """

        return self.softmax(self.agregar_bias(X) @ self.Theta)  # (n, K)

    def predict(self, X):
        """
        Clase de mayor probabilidad.

        Parámetros:
        - X: (n, p)

        Retorna:
        - y_pred: etiquetas (n,)
        """

        return self.clases[np.argmax(self.predict_proba(X), axis=1)]  # (n,)